# Whale Sell-Pull Duration Analysis

**Problem**: Current whale signal detection fires on every snapshot where a whale order disappears — but with ~40μs between ticks, a 3-snapshot absence is just a cancel-resubmit cycle (normal re-pricing). We need to measure absence in **real milliseconds** and find the duration threshold where a whale pulling their sells is signal vs noise.

**Focus**: Whales pulling their **asks (sells)** specifically:
- YES/Up token ask pull → whale doesn't want to sell Up → expects BTC to go up
- NO/Down token ask pull → whale doesn't want to sell Down → expects BTC to go down

**Approach**:
1. For each snapshot, compute total whale ask size across all 5 levels
2. Identify contiguous periods where whale asks = 0 (complete sell-side absence)
3. Measure each absence period in milliseconds
4. Sweep ms-scale thresholds (10ms → 1000ms+) to find where noise ends and signal begins
5. Target: ~20-30 genuine pull events per 4h market (rare, only in volatile markets)

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("../data")
DIR_4H = DATA_DIR / "telonex_btc_4h_book_snapshots"

OVERLAP_START = 1770854400  # Feb 12, 2026 00:00 UTC
OVERLAP_END   = 1772834400  # Mar 6, 2026 22:00 UTC
WHALE_THRESHOLD = 2500      # tokens — catches the 2 dominant MMs

def load_overlap_files():
    files = sorted(DIR_4H.glob("*.parquet"))
    return [(int(f.stem.split("-")[-1]), f) for f in files
            if OVERLAP_START <= int(f.stem.split("-")[-1]) <= OVERLAP_END]

overlap_4h = load_overlap_files()
print(f"4h markets in overlap: {len(overlap_4h)}")

## Step 1: Measure whale ask absence durations

For each token in each 4h market, find every contiguous run where total whale ask size = 0, and measure its duration in milliseconds.

In [ ]:
def compute_whale_ask_presence(df_token: pd.DataFrame) -> np.ndarray:
    """Return boolean array: True where any ask level has size >= WHALE_THRESHOLD."""
    sizes = np.column_stack([df_token[f"ask_size_{i}"].values for i in range(1, 6)])
    return (sizes >= WHALE_THRESHOLD).any(axis=1)


def find_absence_runs(ts_ms: np.ndarray, present: np.ndarray) -> list[dict]:
    """Find contiguous runs where present=False. Return list of dicts with
    start_ms, end_ms, duration_ms, n_ticks (number of absent snapshots)."""
    absent = ~present
    n = len(absent)
    if n == 0 or not absent.any():
        return []

    # Find run boundaries using diff
    padded = np.concatenate([[False], absent, [False]])
    diffs = np.diff(padded.astype(int))
    starts = np.where(diffs == 1)[0]   # absent run starts
    ends = np.where(diffs == -1)[0]     # absent run ends (exclusive)

    runs = []
    for s, e in zip(starts, ends):
        # s, e are indices into the original arrays
        # The absent run covers indices s..e-1
        if s >= n or e > n:
            continue
        start_ms = ts_ms[s]
        # end_ms = timestamp of the first PRESENT snapshot after the run
        # (or last absent snapshot if run goes to the end)
        end_ms = ts_ms[min(e, n - 1)] if e < n else ts_ms[n - 1]
        duration_ms = end_ms - start_ms
        runs.append({
            "start_ms": start_ms,
            "end_ms": end_ms,
            "duration_ms": duration_ms,
            "n_ticks": e - s,
        })
    return runs


def analyze_market_absences(epoch: int, fpath: Path) -> list[dict]:
    """For one 4h market, find all whale ask absence runs for both tokens."""
    df = pq.read_table(str(fpath)).to_pandas()
    all_runs = []

    for label in ["Up", "Down"]:
        tok = df[df.token_label == label].sort_values("exchange_timestamp").reset_index(drop=True)
        if len(tok) < 10:
            continue

        ts = tok["exchange_timestamp"].values
        present = compute_whale_ask_presence(tok)

        # Skip markets where whale is never present (no whale activity)
        if not present.any():
            continue

        runs = find_absence_runs(ts, present)
        for r in runs:
            r["token"] = label
            r["market_epoch"] = epoch
        all_runs.extend(runs)

    return all_runs


# Process all markets
all_runs = []
for epoch, fpath in overlap_4h:
    all_runs.extend(analyze_market_absences(epoch, fpath))

runs_df = pd.DataFrame(all_runs)
print(f"Total whale ask absence runs: {len(runs_df)}")
print(f"\nDuration stats (ms):")
print(runs_df.duration_ms.describe())
print(f"\nDuration stats (seconds):")
print((runs_df.duration_ms / 1000).describe())

## Step 2: Duration distribution — where does noise end and signal begin?

Histogram the absence durations on a log scale to see the natural break between re-pricing flickers and genuine pulls. Also show how many events per market we get at each threshold.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Log-scale histogram of all absence durations
ax = axes[0]
durations_ms = runs_df.duration_ms.values
durations_ms_pos = durations_ms[durations_ms > 0]
ax.hist(np.log10(durations_ms_pos), bins=100, edgecolor="black", linewidth=0.3, alpha=0.7)
ax.set_xlabel("log10(duration_ms)")
ax.set_ylabel("Count")
ax.set_title("Whale Ask Absence Duration Distribution")
# Mark key thresholds
for thresh_ms, color, label in [(10, "orange", "10ms"), (50, "red", "50ms"),
                                  (100, "darkred", "100ms"), (500, "purple", "500ms"),
                                  (1000, "black", "1s")]:
    ax.axvline(np.log10(thresh_ms), color=color, linestyle="--", alpha=0.7, label=label)
ax.legend(fontsize=8)

# 2. Events per market at various thresholds
ax = axes[1]
thresholds_ms = [1, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000]
n_markets = runs_df.market_epoch.nunique()
events_per_market = []
for t in thresholds_ms:
    filtered = runs_df[runs_df.duration_ms >= t]
    avg_per_market = len(filtered) / n_markets if n_markets > 0 else 0
    events_per_market.append(avg_per_market)

ax.plot(thresholds_ms, events_per_market, "o-", linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("Min duration threshold (ms)")
ax.set_ylabel("Avg events per 4h market")
ax.set_title("Signal Count vs Duration Threshold")
ax.axhline(25, color="green", linestyle="--", alpha=0.5, label="~25 target")
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Table of threshold → event counts
ax = axes[2]
ax.axis("off")
table_data = []
for t, epm in zip(thresholds_ms, events_per_market):
    total = len(runs_df[runs_df.duration_ms >= t])
    table_data.append([f"{t}ms", f"{total}", f"{epm:.1f}"])
table = ax.table(cellText=table_data,
                 colLabels=["Threshold", "Total Events", "Per Market"],
                 loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.4)
ax.set_title("Event Counts by Threshold")

plt.tight_layout()
plt.show()

## Step 3: Complementary signal — total book depth drops

Instead of tracking individual whale orders, look at the total ask-side depth across all 5 levels. A sudden drop in total depth (e.g., losing >50% of depth in one tick) could capture the same phenomenon more robustly.

In [ ]:
def compute_depth_drops(df_token: pd.DataFrame, pct_threshold: float = 0.50) -> pd.DataFrame:
    """Find moments where total ask depth drops by >= pct_threshold between consecutive snapshots.
    
    Returns DataFrame with: ts_ms, depth_before, depth_after, pct_drop
    """
    ask_sizes = np.column_stack([df_token[f"ask_size_{i}"].values for i in range(1, 6)])
    total_depth = ask_sizes.sum(axis=1)
    ts = df_token["exchange_timestamp"].values
    
    prev_depth = np.roll(total_depth, 1)
    prev_depth[0] = total_depth[0]
    
    # Avoid div/0
    safe_prev = np.where(prev_depth > 0, prev_depth, 1)
    pct_change = (prev_depth - total_depth) / safe_prev
    
    mask = (pct_change >= pct_threshold) & (prev_depth > 100)  # ignore tiny books
    
    if not mask.any():
        return pd.DataFrame(columns=["ts_ms", "depth_before", "depth_after", "pct_drop"])
    
    idx = np.where(mask)[0]
    return pd.DataFrame({
        "ts_ms": ts[idx],
        "depth_before": prev_depth[idx],
        "depth_after": total_depth[idx],
        "pct_drop": pct_change[idx],
    })


# Analyze depth drops across all markets
all_drops = []
for epoch, fpath in overlap_4h:
    df = pq.read_table(str(fpath)).to_pandas()
    for label in ["Up", "Down"]:
        tok = df[df.token_label == label].sort_values("exchange_timestamp").reset_index(drop=True)
        if len(tok) < 10:
            continue
        drops = compute_depth_drops(tok, pct_threshold=0.50)
        if len(drops) > 0:
            drops["token"] = label
            drops["market_epoch"] = epoch
            all_drops.append(drops)

drops_df = pd.concat(all_drops, ignore_index=True) if all_drops else pd.DataFrame()
print(f"Total 50%+ ask depth drops: {len(drops_df)}")
if len(drops_df) > 0:
    n_mkts = drops_df.market_epoch.nunique()
    print(f"Across {n_mkts} markets = {len(drops_df)/n_mkts:.1f} per market")
    print(f"\nDrop severity:")
    print(drops_df.pct_drop.describe())
    print(f"\nBy token: {drops_df.token.value_counts().to_dict()}")

## Step 4: Correlate with mid-price movement

For each whale ask absence (above threshold) and each depth drop, look at the mid-price change in the N milliseconds following the event. The hypothesis is:
- Up ask pull → mid_price of Up should increase (BTC going up)
- Down ask pull → mid_price of Down should increase (BTC going down)

We measure the subsequent mid-price change at various forward windows (100ms, 500ms, 1s, 5s, 30s).

In [ ]:
FORWARD_WINDOWS_MS = [100, 500, 1000, 2000, 5000, 10000, 30000]

def compute_forward_returns(ts: np.ndarray, mid: np.ndarray, event_ts: np.ndarray,
                            windows_ms: list[int]) -> dict[int, np.ndarray]:
    """For each event timestamp, find the mid-price change at each forward window.
    
    Uses searchsorted for vectorized lookups. Returns dict of window_ms -> array of mid changes.
    """
    results = {}
    for w in windows_ms:
        target_ts = event_ts + w
        # Find index of first snapshot >= target_ts
        idx = np.searchsorted(ts, target_ts, side="left")
        idx = np.clip(idx, 0, len(ts) - 1)
        
        # Mid price at event time
        event_idx = np.searchsorted(ts, event_ts, side="left")
        event_idx = np.clip(event_idx, 0, len(ts) - 1)
        
        mid_at_event = mid[event_idx]
        mid_at_target = mid[idx]
        
        results[w] = mid_at_target - mid_at_event
    return results


def analyze_pull_returns(overlap_files, min_duration_ms: int) -> pd.DataFrame:
    """For whale ask absences >= min_duration_ms, compute forward mid-price returns."""
    all_rows = []
    
    for epoch, fpath in overlap_files:
        df = pq.read_table(str(fpath)).to_pandas()
        
        for label in ["Up", "Down"]:
            tok = df[df.token_label == label].sort_values("exchange_timestamp").reset_index(drop=True)
            if len(tok) < 10:
                continue
            
            ts = tok["exchange_timestamp"].values
            mid = tok["mid_price"].values
            present = compute_whale_ask_presence(tok)
            
            if not present.any():
                continue
            
            runs = find_absence_runs(ts, present)
            # Filter by duration
            event_ts = np.array([r["start_ms"] for r in runs if r["duration_ms"] >= min_duration_ms])
            event_durations = np.array([r["duration_ms"] for r in runs if r["duration_ms"] >= min_duration_ms])
            
            if len(event_ts) == 0:
                continue
            
            fwd = compute_forward_returns(ts, mid, event_ts, FORWARD_WINDOWS_MS)
            
            for i in range(len(event_ts)):
                row = {
                    "ts_ms": event_ts[i],
                    "token": label,
                    "market_epoch": epoch,
                    "absence_duration_ms": event_durations[i],
                }
                for w in FORWARD_WINDOWS_MS:
                    row[f"fwd_{w}ms"] = fwd[w][i]
                all_rows.append(row)
    
    return pd.DataFrame(all_rows)


# Sweep duration thresholds and compute avg forward returns
print("Duration threshold sweep — avg forward mid-price change (ticks = change/0.01):")
print(f"{'Thresh':>8s} {'Events':>7s} {'Per Mkt':>8s}", end="")
for w in FORWARD_WINDOWS_MS:
    print(f" {'fwd_'+str(w)+'ms':>12s}", end="")
print()

for min_dur in [5, 10, 20, 50, 100, 200, 500, 1000]:
    ret_df = analyze_pull_returns(overlap_4h, min_dur)
    if len(ret_df) == 0:
        continue
    n_mkts = ret_df.market_epoch.nunique()
    print(f"{min_dur:>7}ms {len(ret_df):>7d} {len(ret_df)/n_mkts:>8.1f}", end="")
    for w in FORWARD_WINDOWS_MS:
        col = f"fwd_{w}ms"
        # Average: positive = mid moved up (good for Up ask pull, bad for Down ask pull)
        # Split by token to check directionality
        up_mean = ret_df[ret_df.token == "Up"][col].mean() if len(ret_df[ret_df.token == "Up"]) > 0 else 0
        down_mean = ret_df[ret_df.token == "Down"][col].mean() if len(ret_df[ret_df.token == "Down"]) > 0 else 0
        # For a working signal: Up pulls → positive change, Down pulls → positive change
        # (whale pulling asks = they think token is worth more)
        avg_directional = (up_mean + down_mean) / 2  # both should be positive
        print(f" {avg_directional:>12.5f}", end="")
    print()

## Step 5: Detailed analysis at best threshold

Pick the threshold that looks most promising and do a deeper dive:
- Distribution of forward returns (not just mean)
- Win rate if we buy the token whose asks got pulled  
- Comparison across volatile vs calm markets
- Per-market signal count distribution

In [ ]:
# Detailed analysis — we'll run at multiple thresholds and visualize
# Pick a few promising thresholds based on the sweep above

ANALYSIS_THRESHOLDS = [20, 50, 100, 200, 500]

fig, axes = plt.subplots(len(ANALYSIS_THRESHOLDS), 2, figsize=(16, 5 * len(ANALYSIS_THRESHOLDS)))

for row, min_dur in enumerate(ANALYSIS_THRESHOLDS):
    ret_df = analyze_pull_returns(overlap_4h, min_dur)
    if len(ret_df) == 0:
        continue
    
    # Left: histogram of forward returns at 1s and 5s
    ax = axes[row, 0]
    for w, color, label in [(1000, "blue", "1s fwd"), (5000, "orange", "5s fwd")]:
        col = f"fwd_{w}ms"
        ax.hist(ret_df[col].values, bins=50, alpha=0.5, color=color, label=label, density=True)
    ax.axvline(0, color="black", linestyle="--", alpha=0.5)
    ax.set_xlabel("Mid-price change")
    ax.set_ylabel("Density")
    ax.set_title(f"Threshold {min_dur}ms — Forward Returns (n={len(ret_df)})")
    ax.legend()
    
    # Right: per-market signal count distribution
    ax = axes[row, 1]
    per_market = ret_df.groupby("market_epoch").size()
    ax.hist(per_market.values, bins=30, edgecolor="black", alpha=0.7)
    ax.set_xlabel("Signals per 4h market")
    ax.set_ylabel("# Markets")
    ax.set_title(f"Threshold {min_dur}ms — Signals/Market (median={per_market.median():.0f})")
    ax.axvline(25, color="green", linestyle="--", alpha=0.5, label="25 target")
    ax.legend()

plt.tight_layout()
plt.show()

## Step 6: Win rate analysis — buying the pulled token

If the whale pulls their asks for a token, that means they don't want to sell it (they expect it to be worth more). So the trade is: buy that token. 

Win = mid-price increased by at least 1 tick (0.01) within the forward window.
Also compute: what fraction of the time does mid move in the *wrong* direction?

In [ ]:
print("Win rate by duration threshold × forward window")
print("Win = mid moved up ≥ 0.01 (1 tick) after ask pull")
print("Loss = mid moved down ≥ 0.01")
print()

header = f"{'Thresh':>8s} {'N':>6s}"
for w in FORWARD_WINDOWS_MS:
    header += f"  {'WR@'+str(w)+'ms':>10s}"
print(header)

for min_dur in [10, 20, 50, 100, 200, 500, 1000]:
    ret_df = analyze_pull_returns(overlap_4h, min_dur)
    if len(ret_df) == 0:
        continue
    
    line = f"{min_dur:>7}ms {len(ret_df):>6d}"
    for w in FORWARD_WINDOWS_MS:
        col = f"fwd_{w}ms"
        wins = (ret_df[col] >= 0.01).sum()
        total = len(ret_df)
        wr = wins / total * 100 if total > 0 else 0
        line += f"  {wr:>9.1f}%"
    print(line)

print()
print("--- Same but showing avg forward return in ticks (×100) ---")
header = f"{'Thresh':>8s} {'N':>6s}"
for w in FORWARD_WINDOWS_MS:
    header += f"  {'avg@'+str(w)+'ms':>10s}"
print(header)

for min_dur in [10, 20, 50, 100, 200, 500, 1000]:
    ret_df = analyze_pull_returns(overlap_4h, min_dur)
    if len(ret_df) == 0:
        continue
    
    line = f"{min_dur:>7}ms {len(ret_df):>6d}"
    for w in FORWARD_WINDOWS_MS:
        col = f"fwd_{w}ms"
        avg_ticks = ret_df[col].mean() * 100  # in ticks (0.01 = 1 tick)
        line += f"  {avg_ticks:>9.2f}t"
    print(line)

## Step 7: Volatile vs calm market comparison

Split markets by intra-market volatility (range of mid-price within the 4h window). The hypothesis is that pull signals should be concentrated in volatile markets and mostly absent in calm ones.

In [ ]:
# Compute per-market volatility (mid-price range for Up token)
market_vol = {}
for epoch, fpath in overlap_4h:
    df = pq.read_table(str(fpath)).to_pandas()
    up = df[df.token_label == "Up"]
    if len(up) > 0:
        market_vol[epoch] = up.mid_price.max() - up.mid_price.min()

vol_df = pd.DataFrame({"market_epoch": list(market_vol.keys()),
                        "mid_range": list(market_vol.values())})
vol_df = vol_df.sort_values("mid_range")
median_vol = vol_df.mid_range.median()
vol_df["volatile"] = vol_df.mid_range >= median_vol

print(f"Median mid-price range: {median_vol:.3f}")
print(f"Calm markets (< median): {(~vol_df.volatile).sum()}")
print(f"Volatile markets (>= median): {vol_df.volatile.sum()}")

# Now compare signal rates and forward returns
BEST_THRESHOLD = 100  # will adjust after seeing step 4 results

ret_df = analyze_pull_returns(overlap_4h, BEST_THRESHOLD)
if len(ret_df) > 0:
    ret_df = ret_df.merge(vol_df[["market_epoch", "mid_range", "volatile"]], on="market_epoch", how="left")
    
    print(f"\n=== Threshold {BEST_THRESHOLD}ms ===")
    for label, grp in ret_df.groupby("volatile"):
        cat = "VOLATILE" if label else "CALM"
        n_mkts = grp.market_epoch.nunique()
        signals_per_mkt = len(grp) / n_mkts if n_mkts > 0 else 0
        
        print(f"\n  {cat} markets ({n_mkts} markets, {len(grp)} signals, {signals_per_mkt:.1f}/market):")
        for w in [500, 1000, 5000, 30000]:
            col = f"fwd_{w}ms"
            avg = grp[col].mean()
            wr = (grp[col] >= 0.01).mean() * 100
            print(f"    {w:>5}ms fwd: avg={avg:.5f}, WR={wr:.1f}%")

    # Scatter: volatility vs signal count per market
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    per_mkt = ret_df.groupby("market_epoch").agg(
        n_signals=("ts_ms", "size"),
        mid_range=("mid_range", "first"),
        avg_fwd_1s=("fwd_1000ms", "mean"),
    )
    
    ax = axes[0]
    ax.scatter(per_mkt.mid_range, per_mkt.n_signals, alpha=0.5, s=30)
    ax.set_xlabel("Mid-price range (volatility)")
    ax.set_ylabel("Whale ask-pull signals")
    ax.set_title(f"Signal Count vs Volatility (threshold={BEST_THRESHOLD}ms)")
    
    ax = axes[1]
    colors = ["green" if v > 0 else "red" for v in per_mkt.avg_fwd_1s]
    ax.scatter(per_mkt.mid_range, per_mkt.avg_fwd_1s * 100, c=colors, alpha=0.5, s=30)
    ax.axhline(0, color="black", linestyle="--", alpha=0.3)
    ax.set_xlabel("Mid-price range (volatility)")
    ax.set_ylabel("Avg forward return @ 1s (ticks)")
    ax.set_title("Signal Quality vs Volatility")
    
    plt.tight_layout()
    plt.show()

## Step 8: Depth-drop signal correlation

Compare the whale-level ask pull signal with the simpler "total ask depth dropped by X%" signal. Do they fire at the same times? Is one more predictive?

In [ ]:
# Compute forward returns for depth drops
def analyze_depth_drop_returns(overlap_files, pct_threshold: float = 0.50) -> pd.DataFrame:
    """For each depth drop event, compute forward mid-price returns."""
    all_rows = []
    
    for epoch, fpath in overlap_files:
        df = pq.read_table(str(fpath)).to_pandas()
        
        for label in ["Up", "Down"]:
            tok = df[df.token_label == label].sort_values("exchange_timestamp").reset_index(drop=True)
            if len(tok) < 10:
                continue
            
            ts = tok["exchange_timestamp"].values
            mid = tok["mid_price"].values
            
            drops = compute_depth_drops(tok, pct_threshold)
            if len(drops) == 0:
                continue
            
            event_ts = drops["ts_ms"].values
            fwd = compute_forward_returns(ts, mid, event_ts, FORWARD_WINDOWS_MS)
            
            for i in range(len(event_ts)):
                row = {
                    "ts_ms": event_ts[i],
                    "token": label,
                    "market_epoch": epoch,
                    "depth_before": drops.iloc[i]["depth_before"],
                    "depth_after": drops.iloc[i]["depth_after"],
                    "pct_drop": drops.iloc[i]["pct_drop"],
                }
                for w in FORWARD_WINDOWS_MS:
                    row[f"fwd_{w}ms"] = fwd[w][i]
                all_rows.append(row)
    
    return pd.DataFrame(all_rows)


print("=== Depth Drop Signal (total ask depth drops ≥ X%) ===\n")

for pct in [0.30, 0.50, 0.70, 0.90]:
    dd_df = analyze_depth_drop_returns(overlap_4h, pct)
    if len(dd_df) == 0:
        print(f"  {pct*100:.0f}% drop: 0 events")
        continue
    
    n_mkts = dd_df.market_epoch.nunique()
    print(f"  {pct*100:.0f}% drop: {len(dd_df)} events across {n_mkts} markets ({len(dd_df)/n_mkts:.1f}/market)")
    for w in [500, 1000, 5000]:
        col = f"fwd_{w}ms"
        avg = dd_df[col].mean()
        wr = (dd_df[col] >= 0.01).mean() * 100
        print(f"    {w:>5}ms fwd: avg={avg:.5f}, WR={wr:.1f}%")
    print()

## Step 9: Examine individual pull events visually

Pick a few markets with clear whale pull events and plot the mid-price + whale ask presence over time to visually confirm the phenomenon.

In [ ]:
# Pick 3 markets with the most pull events at 100ms threshold
ret_100 = analyze_pull_returns(overlap_4h, 100)
if len(ret_100) > 0:
    top_markets = ret_100.groupby("market_epoch").size().nlargest(3).index.tolist()
    
    fig, axes = plt.subplots(len(top_markets), 1, figsize=(18, 5 * len(top_markets)))
    if len(top_markets) == 1:
        axes = [axes]
    
    for ax, epoch in zip(axes, top_markets):
        fpath = [f for e, f in overlap_4h if e == epoch][0]
        df = pq.read_table(str(fpath)).to_pandas()
        up = df[df.token_label == "Up"].sort_values("exchange_timestamp").reset_index(drop=True)
        
        ts = up["exchange_timestamp"].values
        mid = up["mid_price"].values
        present = compute_whale_ask_presence(up)
        
        # Normalize time to seconds from market start
        t_sec = (ts - ts[0]) / 1000
        
        # Plot mid-price
        ax.plot(t_sec, mid, linewidth=0.5, alpha=0.8, label="Up mid-price", color="blue")
        
        # Shade regions where whale asks are absent
        absent = ~present
        # Find contiguous absent regions
        padded = np.concatenate([[False], absent, [False]])
        diffs = np.diff(padded.astype(int))
        starts = np.where(diffs == 1)[0]
        ends = np.where(diffs == -1)[0]
        
        for s, e in zip(starts, ends):
            if s >= len(t_sec) or e > len(t_sec):
                continue
            duration_ms = ts[min(e, len(ts)-1)] - ts[s]
            if duration_ms >= 100:  # only shade 100ms+ absences
                color = "red" if duration_ms >= 500 else "orange"
                alpha = min(0.5, 0.2 + duration_ms / 5000)
                ax.axvspan(t_sec[s], t_sec[min(e, len(t_sec)-1)], color=color, alpha=alpha)
        
        dt = datetime.fromtimestamp(epoch, tz=timezone.utc)
        ax.set_title(f"Market {dt.strftime('%Y-%m-%d %H:%M')} — Up mid + whale ask absences (orange=100ms+, red=500ms+)")
        ax.set_xlabel("Time (seconds from market start)")
        ax.set_ylabel("Mid-price")
        ax.legend(loc="upper left")
    
    plt.tight_layout()
    plt.show()
else:
    print("No events at 100ms threshold")

## Results Summary

Full analysis run via `whale_pull_duration_runner.py` (v1), `whale_pull_duration_v2.py`, and `whale_pull_duration_v3.py`.

### Duration Distribution
- **Median absence: 508ms**, p75: 2.2s, p95: 15s, p99: 3 minutes
- Even at 10s threshold, we get 77.5 events/market (target: ~25)
- **60s threshold gives ~28/market** — closest to observed rate

### The Signal Direction Works — Weakly
At 2-10s thresholds, individual token ask pulls show **correct directional signal**:
- Same token mid goes UP (+0.00127 avg at 30s for 5s threshold)
- Other token mid goes DOWN (-0.00217 avg at 30s)
- Spread (same - other) = +0.00344 at 30s

But **spread > 0 only 42% of the time** — below 50%, not tradeable alone.

### Best Signal: Uncertain Markets + Depth Drop
| Filter | Events/Market | fwd@5s | fwd@30s | WR@5s |
|--------|--------------|--------|---------|-------|
| 5s pull, mid 0.45-0.55 | ~18 | +0.00226 | +0.00451 | 15.2% |
| 5s pull + 90% depth drop | 26.3 | +0.00172 | +0.00244 | 15.7% |

### What the WR Really Means
WR here is "fraction where mid moved ≥ 1 tick (0.01) in predicted direction." It's 15-16%, not 50%.
This is because most events show **zero mid change** (median fwd = 0.000 at all windows).
The signal is in the tails: 2+ tick moves favor the predicted direction (15.6% vs 12.8% at 30s for 5s pulls).

### Anti-predictive at Extreme Mids
When mid > 0.70 (market already decided), the signal **reverses** — fwd@30s = -0.00275. Makes sense: nothing left to predict.

### Longest Pulls Are Market-Start Noise
The top 20 longest "pulls" (14,000s) are hour-0 events where whales never entered yet — not genuine pulls. Pre-pull whale size = 0.

### Conclusion
The whale sell-pull has a **real but tiny directional edge** at 2-10s thresholds, concentrated in uncertain markets (mid ~0.50) and when combined with 90%+ depth drops (~26 events/market). The effect size (~0.002 ticks avg) is too small to trade as a standalone signal in 5-minute markets, but could serve as a filter or confirmation signal for other strategies.